# Module 4.2: Advanced Attention (MQA & GQA)

Welcome to the final notebook of the LLM Workout dataset! 

In the previous notebook, we mathematically eliminated the dreaded $O(N^2)$ recalculation bottleneck using **KV Caching**. However, we traded *compute time* for *memory space* (RAM/VRAM).

In this notebook, we look at how modern State-of-the-Art models (like Llama 2 and Llama 3) modify the Attention mechanism to permanently fix the memory growth problem using **Multi-Query Attention (MQA)** and **Grouped-Query Attention (GQA)**.

## 1. The Memory Bottleneck of Multi-Head Attention (MHA)

In standard Multi-Head Attention (which we built in Module 2), each Attention "Head" gets its own dedicated: 
- `Q` (Query) matrix
- `K` (Key) matrix
- `V` (Value) matrix

When doing KV-Caching, this means **every single head** is saving its own `K` and `V` vectors to memory for every single token generated. If you have 32 Heads (like Llama 2 7B) or 128 Heads (GPT-4), you are duplicating memory saving across *all* those heads.

```mermaid
graph LR
    A[Head 1] -->|Caches| K1[Key 1]
    A -->|Caches| V1[Value 1]
    B[Head 2] -->|Caches| K2[Key 2]
    B -->|Caches| V2[Value 2]
    C[Head N] -->|Caches| KN[Key N]
    C -->|Caches| VN[Value N]
```

## 2. Multi-Query Attention (MQA)

**MQA** proposes a radical solution to the memory bloat: What if *every* Attention Head still gets its own Query (Q), but they all **share a single Key (K) and a single Value (V)** projection?

```mermaid
graph LR
    A[Head 1 Query] -->|Shares| K[Shared Key Projection]
    B[Head 2 Query] -->|Shares| K
    C[Head N Query] -->|Shares| K
    
    A -->|Shares| V[Shared Value Projection]
    B -->|Shares| V
    C -->|Shares| V
```

### Why do this?
If we only have 1 `K` vector and 1 `V` vector instead of 32, our KV Cache is suddenly **32 times smaller**! We can fit a context size of 100,000 tokens on a single GPU.

*Note: This usually hurts model quality slightly because heads can't express diverse Key/Value concepts, but the memory savings are monstrous.*

In [ ]:
import torch
import torch.nn as nn

class MultiQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        # The magic of MQA:
        # Query still projects to the full d_model (each head gets its own slice)
        self.W_q = nn.Linear(d_model, d_model)
        
        # Key and Value ONLY project to the size of a SINGLE head! (Shared!)
        self.W_k = nn.Linear(d_model, self.d_head)
        self.W_v = nn.Linear(d_model, self.d_head)

    def forward(self, x):
        batch, seq_len, d_model = x.shape
        
        # Shape Q: (Batch, Seq_Len, Num_Heads, Head_Dim)
        Q = self.W_q(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        
        # Shape K, V: (Batch, Seq_Len, 1, Head_Dim) -> Notice the "1"!
        # We simulate the "sharing" by creating a pseudo num_heads dimension of size 1
        K = self.W_k(x).view(batch, seq_len, 1, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_len, 1, self.d_head).transpose(1, 2)
        
        # PyTorch will automatically stretch (broadcast) the "1" dimension of K and V 
        # to match the "num_heads" dimension of Q during the dot product!
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        
        # Output shape matches standard MHA: (Batch, Num_Heads, Seq_Len, Head_Dim)
        output = torch.matmul(weights, V)
        return output
        
mqa_layer = MultiQueryAttention(d_model=512, num_heads=8)
dummy_input = torch.randn(2, 10, 512)
out = mqa_layer(dummy_input)
print(f"MQA Output shape: {out.shape} -> Successfully calculated Attention!")

## 3. Grouped-Query Attention (GQA)

If normal Attention (MHA) has the best quality but terrible memory, and MQA has amazing memory but worse quality, what do we do?

**GQA** is the perfect middle ground. Instead of 1 shared K/V, or 32 unshared K/Vs, we split the difference. If we have 32 queries, let's group them into 8 groups of 4. Each group gets its own shared K and V.

- **MHA** (Full Quality): 32 Qs, 32 Ks, 32 Vs
- **MQA** (Full Memory): 32 Qs, 1 K, 1 V
- **GQA** (Goldilocks): 32 Qs, 8 Ks, 8 Vs. (This is what Llama 2 & 3 exclusively use!).

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_groups=2):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.d_head = d_model // num_heads
        
        # Validate division
        assert num_heads % num_kv_groups == 0, "Heads must be evenly grouped!"
        self.heads_per_group = num_heads // num_kv_groups

        self.W_q = nn.Linear(d_model, d_model)
        
        # K and V project to the size of the GROUPS, not the individual heads!
        d_kv = num_kv_groups * self.d_head
        self.W_k = nn.Linear(d_model, d_kv)
        self.W_v = nn.Linear(d_model, d_kv)

    def forward(self, x):
        batch, seq_len, d_model = x.shape
        
        # Shape Q: (Batch, Seq_Len, Num_Heads, Head_Dim)
        Q = self.W_q(x).view(batch, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        
        # Shape K, V: (Batch, Seq_Len, Groups, Head_Dim)
        K = self.W_k(x).view(batch, seq_len, self.num_kv_groups, self.d_head).transpose(1, 2)
        V = self.W_v(x).view(batch, seq_len, self.num_kv_groups, self.d_head).transpose(1, 2)
        
        # To do the dot product natively, we can use `torch.repeat_interleave`
        # to physically copy the K and V tensors so that they stretch to match Num_Heads.
        # Ex: If we have 4 groups, and 8 heads total. Group 1 repeats twice, Group 2 repeats twice...
        # Result: 8 K and V tensors, ready for standard Multi-Head Attention!
        
        K_expanded = torch.repeat_interleave(K, repeats=self.heads_per_group, dim=1)
        V_expanded = torch.repeat_interleave(V, repeats=self.heads_per_group, dim=1)
        
        # Regular Attention Math!
        scores = torch.matmul(Q, K_expanded.transpose(-2, -1)) / (self.d_head ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        
        output = torch.matmul(weights, V_expanded)
        return output
        
# 8 Queries, grouped into pairs (4 Groups of K and V!)
gqa_layer = GroupedQueryAttention(d_model=512, num_heads=8, num_kv_groups=4)
dummy_input = torch.randn(2, 10, 512)
out = gqa_layer(dummy_input)
print(f"GQA Output shape: {out.shape} -> Look at that Grouped Attention go!")

## 🎓 Journey Complete!

You have officially coded the mathematical and scalable foundation of the **Transformers architecture** from purely basic matrices straight down to the bleeding edge optimizations powering the largest foundation models in the world today.

#### **What's Next for you?**
1. **Pre-training:** Writing distributed training loops across clusters of GPUs.
2. **Fine-Tuning:** Instruction tuning (SFT) and RLHF (DPO/PPO) to make the model chat nicely.
3. **Inference Scaling:** Integrating optimized backend libraries like vLLM to serve these large architectures dynamically.